# Evaluación y análisis

Ejecuta, sobre los pesos entrenados, todos los análisis de la Pregunta 3 y muestra sus resultados:

| Entregable | Script | Salida |
|---|---|---|
| E2 · mAP por especie | `src/eval/evaluate.py` | `results/eval_*.csv` |
| E3 · cis vs trans y falsas alarmas en vacías | `src/eval/evaluate.py`, `src/eval/threshold_analysis.py` | `results/eval_summary.csv`, `results/fp_vacias.csv` |
| E3 · EigenCAM | `src/eval/eigencam.py` | `results/e3_eigencam_*.csv`, figuras `e3_eigencam_*.png` |
| E4 · tamaño de caja | `src/robustness/size_eval.py` | `results/e4_tamano_*.csv` |
| E4 · oclusión | `src/robustness/occlusion_eval.py` | `results/e4_oclusion_full_test.csv` |
| E5 · monitoreo en la cámara 46 | `src/monitoring/report.py` | `results/e5_*_cam46.csv`, figura `e5_actividad_cam46.png` |
| E6 · umbral de confianza | `src/eval/threshold_analysis.py` | `results/umbral_*.csv` |

Es lo mismo que hacen los pasos 3 a 7 de `scripts/run_all.sh`. Las celdas de la sección **Resultados** solo leen `results/` y `reports/figures/`, así que se pueden ejecutar sin GPU con los archivos del repo.

## Preparar la máquina (Colab)

Se ejecuta `colab_setup.ipynb` completo (extracción de imágenes y etiquetas YOLO). Los pesos se leen de Drive; también están en el release `pesos-v1` del repo.

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive') and os.path.isdir('/content/drive'):
    import shutil
    shutil.rmtree('/content/drive')  # restos locales de una sesión anterior impiden montar Drive
drive.mount('/content/drive')
REPO_DIR = '/content/redes-parcial-p3'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Roxana-Vargas/redes-parcial-p3.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
%run notebooks/colab_setup.ipynb

In [ ]:
D = '/content/drive/MyDrive/redes-parcial-p3'
RUNS, EVAL = f'{D}/runs', f'{D}/eval'
FULL, ECCV, BG = (f'{RUNS}/{m}/weights/best.pt' for m in ['full_yolov8s', 'eccv_yolov8s', 'full_bg18786'])

## E2 y E3 · Evaluación en test

In [ ]:
!python src/eval/evaluate.py --weights {FULL} --data data/yolo/data_full.yaml --split full_test --model full_yolov8s --project {EVAL}
!python src/eval/evaluate.py --weights {BG} --data data/yolo/data_full.yaml --split full_test --model full_bg18786 --project {EVAL}
for split in ['eccv_cis_test', 'eccv_trans_test']:
    !python src/eval/evaluate.py --weights {ECCV} --data data/yolo/data_eccv.yaml --split {split} --model eccv_yolov8s --project {EVAL}
    !python src/eval/evaluate.py --weights {FULL} --data data/yolo/data_full.yaml --split {split} --model full_yolov8s --project {EVAL}
!python src/eval/threshold_analysis.py --eval-dir {EVAL}

## E3 · EigenCAM

Capa 9 (SPPF, final del backbone) para los dos modelos y capa 15 (neck, P3) para el modelo E3.

In [ ]:
!python src/eval/eigencam.py --weights {ECCV} --model eccv_yolov8s
!python src/eval/eigencam.py --weights {FULL} --model full_yolov8s
!python src/eval/eigencam.py --weights {ECCV} --model eccv_yolov8s_capa15 --layer 15

## E4 · Tamaño de caja y oclusión

La oclusión se anotó a mano en 298 cajas de `full_test` (`annotations/occlusion_test.csv`, con `src/robustness/annotate_occlusion.py`).

In [ ]:
!python src/robustness/size_eval.py --pred {EVAL}/full_yolov8s_full_test/predictions.json --split full_test --model full_yolov8s
!python src/robustness/occlusion_eval.py --eval-dir {EVAL} --models full_yolov8s full_bg18786
!python src/robustness/occlusion_figure.py --img-dir data/raw/cct_images

## E5 · Monitoreo en la cámara 46

In [ ]:
!python src/monitoring/report.py --weights {FULL} --location 46

## Figuras y respaldo en Drive

In [ ]:
!python src/eval/figures.py
!mkdir -p {D}/resultados {D}/figuras
!cp results/*.csv {D}/resultados/
!cp reports/figures/e*.png reports/figures/umbral_*.png {D}/figuras/

# Resultados

Desde aquí solo se leen archivos del repo (se puede ejecutar en local).

In [ ]:
import pandas as pd
from IPython.display import Image, display


def fig(name, width=800):
    display(Image(filename=f'reports/figures/{name}', width=width))

## E2 · mAP por especie en cámaras no vistas (`full_test`)

In [ ]:
display(pd.read_csv('results/eval_summary.csv'))
display(pd.read_csv('results/eval_full_yolov8s_full_test.csv').sort_values('cajas_train', ascending=False))
fig('e2_ap_por_especie.png')

## E3 · Cámaras vistas (cis) frente a cámaras nuevas (trans)

In [ ]:
fig('e3_cis_vs_trans.png')
fig('e3_brecha_especie.png')
display(pd.read_csv('results/fp_vacias.csv'))
fig('e3_fp_vacias.png', 500)
display(pd.read_csv('results/e3_eigencam_resumen.csv'))
fig('e3_eigencam_eccv_yolov8s.png', 900)

## E4 · Robustez por tamaño y oclusión

In [ ]:
display(pd.read_csv('results/e4_tamano_full_yolov8s_full_test.csv'))
fig('e4_tamano.png', 500)
display(pd.read_csv('results/e4_oclusion_full_test.csv'))
fig('e4_oclusion_recall.png', 600)
fig('e4_oclusion_ejemplos.png', 900)

## E5 · Reporte de actividad de la cámara 46

In [ ]:
display(pd.read_csv('results/e5_resumen_cam46.csv'))
display(pd.read_csv('results/e5_reporte_cam46.csv'))
fig('e5_actividad_cam46.png', 900)

## E1 y E6 · Ablación de vacías y umbral de confianza

In [ ]:
display(pd.read_csv('results/umbral_full_yolov8s_full_test.csv'))
fig('umbral_compromiso.png', 900)